# FFXI Telemetry Lab analysis

## tl;dr

- The frozen backfill contains **1,245 completed fights** and
  **47,094 MCP operations**.
- MCP operation success is **98.35%**; the attack rejection rate is
  **22.90%**.
- Navigation probes reached their destination in **76 of 138**
  attempts (**55.1%**), with **42 partial-progress** and
  **20 stalled** outcomes.
- The latest ingestion session quarantined **0 malformed rows** and
  reconciled exactly to its frozen source boundaries.
- Historical EXP, deaths, and recoveries are unavailable because event history
  does not carry those counters. They become available only during state-observer
  coverage.


## Context & Methods

This companion notebook reads only tested Gold models from the local DuckDB
database. It never queries raw payloads or the gameplay project. Historical Git
SHAs are inferred from the most recent commit at or before each event timestamp;
observer and incremental rows use the HEAD observed at ingestion.

### Key Assumptions

- `fight_complete` is the authoritative historical fight count.
- A successful collision probe has outcome `arrived`; `partial_progress` and
  `stalled` remain separate outcomes.
- Service-teleport operations are a dependency proxy, not proof of a navigation
  cause.
- Job is unavailable in the current event/state contracts.


## Data

### 1. Load tested Gold models

In [ ]:
from pathlib import Path

import duckdb
import plotly.express as px

ROOT = Path.cwd()
DB_PATH = ROOT / "data" / "warehouse" / "telemetry.duckdb"
assert DB_PATH.is_file(), "Run backfill, prepare-warehouse, and dbt build first."
connection = duckdb.connect(str(DB_PATH), read_only=True)


### 2. Verify data quality

In [ ]:
quality = connection.execute(
    '''
    select source, bronze_rows, distinct_event_ids, null_event_times,
           duplicate_event_ids, latest_session_malformed_rows,
           latest_session_reconciled
    from gold.gold_data_quality
    order by source
    '''
).df()
assert quality["duplicate_event_ids"].sum() == 0
assert quality["null_event_times"].sum() == 0
assert quality["latest_session_reconciled"].all()
quality


## Results

### 3. Autonomous progression

In [ ]:
progress = connection.execute(
    '''
    select event_date,
           sum(completed_fights) completed_fights,
           sum(target_levels_reached) target_levels_reached,
           sum(objective_milestones) objective_milestones,
           sum(exp_earned) exp_earned,
           max(exp_metric_quality) exp_metric_quality
    from gold.gold_autonomous_progression
    group by event_date
    order by event_date
    '''
).df()
progress


In [ ]:
progress_plot = progress.melt(
    id_vars=["event_date"],
    value_vars=["completed_fights", "target_levels_reached", "objective_milestones"],
    var_name="metric",
    value_name="count",
)
fig = px.line(
    progress_plot,
    x="event_date",
    y="count",
    color="metric",
    markers=True,
    title="Autonomous progression by day",
)
fig.update_layout(xaxis_title=None, yaxis_title="Events", template="plotly_white")
fig.show()


### 4. Combat reliability

In [ ]:
combat = connection.execute(
    '''
    select event_date,
           sum(completed_fights) completed_fights,
           sum(proactive_engagements) proactive_engagements,
           sum(reactive_engagements) reactive_engagements,
           sum(attack_issued) attack_issued,
           sum(attack_rejections) attack_rejections,
           sum(target_cycle_errors) target_cycle_errors,
           sum(weapon_skills) weapon_skills,
           sum(job_abilities) job_abilities,
           sum(combat_spells) combat_spells,
           max(aggro_response_p95_ms) aggro_response_p95_ms,
           max(handoff_queue_p95_ms) handoff_queue_p95_ms
    from gold.gold_combat_reliability
    group by event_date
    order by event_date
    '''
).df()
combat["attack_rejection_rate"] = (
    combat["attack_rejections"]
    / (combat["attack_issued"] + combat["attack_rejections"])
)
combat


In [ ]:
engagements = combat.melt(
    id_vars=["event_date"],
    value_vars=["proactive_engagements", "reactive_engagements"],
    var_name="mode",
    value_name="engagements",
)
fig = px.bar(
    engagements,
    x="event_date",
    y="engagements",
    color="mode",
    barmode="stack",
    title="Engagement mix by day",
    color_discrete_sequence=["#2563EB", "#D4A72C"],
)
fig.update_layout(xaxis_title=None, yaxis_title="Engagements", template="plotly_white")
fig.show()


### 5. Navigation performance

In [ ]:
navigation = connection.execute(
    '''
    select event_date,
           sum(collision_probes) collision_probes,
           sum(successful_collision_probes) arrived,
           sum(partial_progress_probes) partial_progress,
           sum(stalled_probes) stalled,
           sum(camp_relocations) camp_relocations,
           sum(zone_transitions) zone_transitions,
           sum(line_of_sight_nudges) line_of_sight_nudges,
           sum(navigation_failures) navigation_failures,
           sum(navigation_retries) navigation_retries,
           max(service_teleport_operations) service_teleport_operations
    from gold.gold_navigation_performance
    group by event_date
    order by event_date
    '''
).df()
navigation


In [ ]:
outcomes = navigation[["arrived", "partial_progress", "stalled"]].sum().reset_index()
outcomes.columns = ["outcome", "attempts"]
fig = px.bar(
    outcomes,
    x="attempts",
    y="outcome",
    orientation="h",
    title="Collision probe outcomes",
    color_discrete_sequence=["#2563EB"],
)
fig.update_layout(xaxis_title="Attempts", yaxis_title=None, template="plotly_white")
fig.show()


### 6. MCP reliability and Git attribution

In [ ]:
mcp = connection.execute(
    '''
    select operation,
           sum(operation_count) operation_count,
           sum(failed_operations) failed_operations,
           sum(successful_operations)::double / nullif(sum(operation_count), 0)
             as success_rate,
           max(duration_p95_ms) duration_p95_ms
    from gold.gold_mcp_operation_reliability
    group by operation
    order by operation_count desc
    limit 15
    '''
).df()
mcp


In [ ]:
by_commit = connection.execute(
    '''
    select substr(source_git_sha, 1, 8) source_git_sha,
           git_sha_provenance,
           completed_fights,
           attack_rejection_rate,
           mcp_operations,
           mcp_failure_rate,
           navigation_failures
    from gold.gold_performance_by_git_commit
    order by first_event_date, source_git_sha
    '''
).df()
by_commit


## Takeaways

1. **Control is broadly reliable:** 47,094 operations completed at
   98.35% success across the backfill window.
2. **Combat has a measurable retry cost:** 541 rejected attacks against
   1,821 issued attacks produce a 22.90% rejection rate.
3. **Navigation is mixed rather than binary:** 76 probes arrived,
   42 made partial progress, and 20 stalled. Treating partial
   progress as success would overstate reliability.
4. **Coverage labels matter:** no historical EXP/death/recovery series is claimed.
   Those counters begin only when the state observer is running.
5. **Attribution is directional:** historical commit comparisons are useful for
   investigation but remain inferred, not authoritative.


In [ ]:
connection.close()